# Improved Arabic Handwritten Word Recognition

**Key improvements:**
1. Single-class YOLO for detection only
2. Custom CNN (not MobileNetV2) for 64×64 grayscale characters
3. Proper label alignment with actual dataset classes
4. Right-to-left sorting for Arabic reading order

## 1. Configuration

In [1]:
import os, shutil, random, numpy as np, cv2
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT       = Path(r"F:\bach2\Arabic-Handwritten-Characters-Recognition-using-CNN")
AHAWP_DIR  = ROOT / "isolated_alphabets_per_alphabet"   # isolated chars for CNN
MY_DATASET = ROOT / "my_dataset"                         # Label Studio YOLO export
IMG_SIZE   = 64

## 2. Convert YOLO labels to single-class
Rewrite all label files so YOLO only learns to **locate** characters (class 0).

In [2]:
def convert_to_single_class(dataset_dir):
    """Rewrite all YOLO label files to use class_id=0 (single class: 'character')."""
    for split in ['train', 'val']:
        labels_dir = dataset_dir / split / 'labels'
        if not labels_dir.exists():
            print(f"  Skipping {split} (not found)")
            continue
        count = 0
        for txt_file in labels_dir.glob('*.txt'):
            lines = txt_file.read_text().splitlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = '0'
                    new_lines.append(' '.join(parts))
            txt_file.write_text('\n'.join(new_lines) + '\n')
            count += 1
        print(f"  Converted {count} label files in {split}/")

print("Converting YOLO labels to single-class...")
convert_to_single_class(MY_DATASET)

Converting YOLO labels to single-class...
  Converted 60 label files in train/
  Converted 13 label files in val/


## 3. Write single-class data.yaml

In [4]:
yaml_path = MY_DATASET / 'data.yaml'
with open(yaml_path, 'w') as f:
    f.write(f"path: {MY_DATASET.as_posix()}\n")
    f.write("train: train/images\n")
    f.write("val: val/images\n")
    f.write("nc: 1\n")
    f.write("names:\n")
    f.write("  - character\n")
print(f"Saved single-class data.yaml: {yaml_path}")

Saved single-class data.yaml: F:\bach2\Arabic-Handwritten-Characters-Recognition-using-CNN\my_dataset\data.yaml


## 4. Build & Train the CNN for Character Classification

### 4a. Imports & Load Dataset

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPool2D, Dense, Dropout,
    BatchNormalization, Flatten, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

tf.random.set_seed(SEED)

def load_isolated_dataset(alphabets_dir, img_size=IMG_SIZE):
    """Load all isolated character images. Use folder name as label directly."""
    images, labels = [], []
    for subfolder in sorted(alphabets_dir.iterdir()):
        if not subfolder.is_dir():
            continue
        label = subfolder.name
        for img_path in subfolder.glob("*.png"):
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (img_size, img_size))
            images.append(img)
            labels.append(label)
    
    X = np.array(images, dtype='float32') / 255.0
    X = X[..., np.newaxis]
    y = np.array(labels)
    print(f"Loaded {len(X)} images across {len(set(labels))} classes")
    return X, y

X_all, y_all = load_isolated_dataset(AHAWP_DIR)

le = LabelEncoder()
y_encoded = le.fit_transform(y_all)
NUM_CLASSES = len(le.classes_)
print(f"Classes found: {NUM_CLASSES}")
print(f"Classes: {list(le.classes_)}")

np.save("label_classes.npy", le.classes_)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_encoded, test_size=0.25, random_state=SEED, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

Loaded 53199 images across 65 classes
Classes found: 65
Classes: [np.str_('ain_begin'), np.str_('ain_end'), np.str_('ain_middle'), np.str_('ain_regular'), np.str_('alif_end'), np.str_('alif_hamza'), np.str_('alif_regular'), np.str_('beh_begin'), np.str_('beh_end'), np.str_('beh_middle'), np.str_('beh_regular'), np.str_('dal_end'), np.str_('dal_regular'), np.str_('feh_begin'), np.str_('feh_end'), np.str_('feh_middle'), np.str_('feh_regular'), np.str_('heh_begin'), np.str_('heh_end'), np.str_('heh_middle'), np.str_('heh_regular'), np.str_('jeem_begin'), np.str_('jeem_end'), np.str_('jeem_middle'), np.str_('jeem_regular'), np.str_('kaf_begin'), np.str_('kaf_end'), np.str_('kaf_middle'), np.str_('kaf_regular'), np.str_('lam_alif'), np.str_('lam_begin'), np.str_('lam_end'), np.str_('lam_middle'), np.str_('lam_regular'), np.str_('meem_begin'), np.str_('meem_end'), np.str_('meem_middle'), np.str_('meem_regular'), np.str_('noon_begin'), np.str_('noon_end'), np.str_('noon_middle'), np.str_('noo

### 4b. Build & Train CNN

In [6]:
def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=NUM_CLASSES):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(32, 3, padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(32, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPool2D(2),
        Dropout(0.25),
        
        Conv2D(64, 3, padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(64, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPool2D(2),
        Dropout(0.25),
        
        Conv2D(128, 3, padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(128, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPool2D(2),
        Dropout(0.25),
        
        Conv2D(256, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPool2D(2),
        Dropout(0.25),
        
        Flatten(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax'),
    ], name='ArabicCharCNN')
    return model

model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "ArabicCharCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 2,832,161 (10.80 MB)

 Trainable params: 2,829,729 (10.79 MB)

 Non-trainable params: 2,432 (9.50 KB)

### 4c. Data Augmentation & Training

In [7]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.05,
    fill_mode='nearest'
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint('best_arabic_cnn.keras', monitor='val_accuracy', save_best_only=True),
]

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc:.4f} ({acc*100:.2f}%)")
model.save('arabic_cnn_final.keras')

Epoch 1/50
624/624 ━━━━━━━━━━━━━━━━━━━━ 300s 473ms/step - accuracy: 0.2116 - loss: 3.1691 - val_accuracy: 0.0217 - val_loss: 9.5231 - learning_rate: 0.0010
Epoch 2/50
624/624 ━━━━━━━━━━━━━━━━━━━━ 300s 480ms/step - accuracy: 0.6091 - loss: 1.2983 - val_accuracy: 0.0337 - val_loss: 23.6217 - learning_rate: 0.0010
Epoch 3/50
624/624 ━━━━━━━━━━━━━━━━━━━━ 301s 482ms/step - accuracy: 0.7288 - loss: 0.8851 - val_accuracy: 0.4998 - val_loss: 1.9828 - learning_rate: 0.0010
Epoch 4/50
624/624 ━━━━━━━━━━━━━━━━━━━━ 302s 484ms/step - accuracy: 0.7768 - loss: 0.7229 - val_accuracy: 0.1585 - val_loss: 8.1683 - learning_rate: 0.0010
Epoch 5/50
624/624 ━━━━━━━━━━━━━━━━━━━━ 301s 483ms/step - accuracy: 0.8039 - loss: 0.6395 - val_accuracy: 0.2427 - val_loss: 4.8856 - learning_rate: 0.0010
Epoch 6/50
624/624 ━━━━━━━━━━━━━━━━━━━━ 307s 491ms/step - accuracy: 0.8245 - loss: 0.5769 - val_accuracy: 0.0259 - val_loss: 31.1357 - learning_rate: 0.0010
Epoch 7/50
624/624 ━━━━━━━━━━━━━━━━━━━━ 298s 478ms/step - accu

## 5. Train YOLO as Single-Class Detector

In [8]:
from ultralytics import YOLO

yolo_model = YOLO('yolov8s.pt')

yolo_results = yolo_model.train(
    data=str(yaml_path),
    epochs=150,
    imgsz=640,
    batch=4,
    patience=30,
    device='cpu',         # change to 'cuda' if you have GPU
    project=str(ROOT / 'my_yolo'),
    name='char_detector_v2',
    exist_ok=True,
    degrees=10.0,
    translate=0.15,
    scale=0.4,
    shear=3.0,
    flipud=0.0,
    fliplr=0.0,           # don't flip Arabic text!
    mosaic=0.5,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    erasing=0.3,
)

YOLO_WEIGHTS = ROOT / 'my_yolo' / 'char_detector_v2' / 'weights' / 'best.pt'
print(f"\nYOLO training complete! Weights: {YOLO_WEIGHTS}")

New https://pypi.org/project/ultralytics/8.4.25 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.11.0 torch-2.10.0+cpu CPU (Intel Core i7-10750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\bach2\Arabic-Handwritten-Characters-Recognition-using-CNN\my_dataset\data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.3, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0

## 6. Inference Pipeline
YOLO detects → CNN classifies → sort right-to-left

In [9]:
cnn_model     = tf.keras.models.load_model('best_arabic_cnn.keras')
yolo_detector = YOLO(str(YOLO_WEIGHTS))
label_classes = np.load('label_classes.npy', allow_pickle=True)

def preprocess_for_cnn(crop_bgr, img_size=IMG_SIZE):
    """Preprocess a YOLO crop for the CNN classifier."""
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    coords = cv2.findNonZero(thresh)
    if coords is None:
        return None
    x, y, w, h = cv2.boundingRect(coords)
    roi = thresh[y:y+h, x:x+w]
    
    side = int(max(h, w) * 1.3)
    canvas = np.zeros((side, side), dtype='uint8')
    ox, oy = (side - w) // 2, (side - h) // 2
    canvas[oy:oy+h, ox:ox+w] = roi
    
    resized = cv2.resize(canvas, (img_size, img_size), interpolation=cv2.INTER_AREA)
    normalized = resized.astype('float32') / 255.0
    return normalized[..., np.newaxis]


def recognize_word(image_path, conf_threshold=0.25, iou_threshold=0.4):
    """
    1. YOLO finds character bounding boxes
    2. CNN classifies each crop
    3. Sort right-to-left for Arabic
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")
    
    H, W = img.shape[:2]
    annotated = img.copy()
    
    results = yolo_detector(img, conf=conf_threshold, iou=iou_threshold, verbose=False)[0]
    boxes = results.boxes
    
    if len(boxes) == 0:
        print("No characters detected.")
        return [], annotated
    
    detections = []
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        det_conf = float(box.conf[0])
        
        x1c, y1c = max(0, x1), max(0, y1)
        x2c, y2c = min(W, x2), min(H, y2)
        crop = img[y1c:y2c, x1c:x2c]
        if crop.size == 0:
            continue
        
        proc = preprocess_for_cnn(crop)
        if proc is None:
            continue
        
        inp = np.expand_dims(proc, axis=0)
        preds = cnn_model.predict(inp, verbose=0)[0]
        cnn_idx = np.argmax(preds)
        cnn_conf = float(preds[cnn_idx])
        label = str(label_classes[cnn_idx])
        
        base_letter = label.rsplit('_', 1)[0] if '_' in label else label
        detections.append((x1, y1, x2, y2, label, base_letter, cnn_conf, det_conf))
    
    detections.sort(key=lambda d: -d[0])  # Right-to-left
    
    recognized_full = []
    recognized_base = []
    for (x1, y1, x2, y2, label, base, cnn_conf, det_conf) in detections:
        recognized_full.append(label)
        recognized_base.append(base)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 200, 0), 2)
        display = f"{base} ({cnn_conf:.0%})"
        cv2.putText(annotated, display,
                    (x1, max(0, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 0), 1)
    
    return recognized_base, annotated

## 7. Test on Your Images

In [10]:
import matplotlib.pyplot as plt

test_images = list((MY_DATASET / 'val' / 'images').glob('*.*'))
for img_path in test_images[:5]:
    chars, annotated = recognize_word(img_path)
    print(f"\n{img_path.name}: {' '.join(chars)}")
    
    plt.figure(figsize=(10, 4))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f"Detected: {' '.join(chars)}")
    plt.show()


ce0882a8-WhatsApp_Image_2026-03-23_at_2.21.24_AM_8.jpeg: sad meem meem


<Figure size 1000x400 with 1 Axes>


d272ef07-WhatsApp_Image_2026-03-23_at_2.21.26_AM_25.jpeg: heh heh heh


<Figure size 1000x400 with 1 Axes>


da1fee74-WhatsApp_Image_2026-03-23_at_2.21.26_AM_18.jpeg: heh meem heh


<Figure size 1000x400 with 1 Axes>


dd921431-WhatsApp_Image_2026-03-23_at_2.21.26_AM_20.jpeg: heh meem heh meem heh


<Figure size 1000x400 with 1 Axes>


e05f66fa-WhatsApp_Image_2026-03-23_at_2.21.26_AM_9.jpeg: heh heh heh


<Figure size 1000x400 with 1 Axes>

## Tips to Improve Further

1. **More YOLO data**: Label 200-300 word images (single-class boxes are fast to draw!)
2. **Synthetic data**: Compose isolated characters into fake words with auto-generated labels
3. **CNN improvements**: Try larger IMG_SIZE (96/128), add elastic distortion augmentation
4. **Post-processing**: Use Arabic language model to correct unlikely character sequences

In [20]:

import pandas as pd
for i in [4,5,6,7]:
    csv_path = f'runs/detect/train{i}/results.csv'
    try:
        df = pd.read_csv(csv_path)
        print(f"train{i} — final mAP50: {df['metrics/mAP50(B)'].iloc[-1]:.4f}")
    except: pass


train4 — final mAP50: 0.0400
train5 — final mAP50: 0.4667
train6 — final mAP50: 0.7572
train7 — final mAP50: 0.0591


In [ ]:
from tensorflow.keras.models import load_model

# --- Test: Recognize a handwritten Arabic word ---
TEST_IMAGE = "word_test.png"
YOLO_MODEL = r"F:\bach2\Arabic-Handwritten-Characters-Recognition-using-CNN\my_yolo\char_detector_v2\weights\best.pt"
CNN_MODEL = "best_arabic_cnn.keras"

yolo = YOLO(YOLO_MODEL)
cnn  = load_model(CNN_MODEL)

img = cv2.imread(TEST_IMAGE)
results = yolo.predict(img, conf=0.25)[0]
boxes = results.boxes.xyxy.cpu().numpy()

# Sort right-to-left (Arabic reading order)
boxes = boxes[boxes[:, 0].argsort()[::-1]]

word = ""
for box in boxes:
    x1, y1, x2, y2 = map(int, box[:4])
    crop = img[y1:y2, x1:x2]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    resized = cv2.resize(thresh, (64, 64))
    inp = (resized.astype('float32') / 255.0).reshape(1, 64, 64, 1)
    pred = cnn.predict(inp, verbose=0)
    word += CLASS_NAMES[np.argmax(pred)]

print(f"Recognized word: {word}")



0: 192x416 (no detections), 22.2ms
Speed: 1.1ms preprocess, 22.2ms inference, 0.5ms postprocess per image at shape (1, 3, 192, 416)
Recognized word: 
